# L05 – Lösningar
Kräver att du har kört `%%writefile`-cellerna i L05, så att `dlkit` finns.

In [ ]:
import sys
sys.path.insert(0, "..")
import torch
import torch.nn as nn
from dlkit.seed import set_seed
from dlkit.data import fashion_mnist_loaders
from dlkit.train import fit

device = "cuda" if torch.cuda.is_available() else "cpu"

### Övning 1
Varje `Linear(in, out)` har `in*out` vikter och `out` bias:
- 784·100 + 100 = 78 500
- 100·50 + 50 = 5 050
- 50·10 + 10 = 510

Summan är **84 060**.

In [ ]:
my_mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 100), nn.ReLU(),
    nn.Linear(100, 50), nn.ReLU(),
    nn.Linear(50, 10),
)
my_count_by_hand = (784 * 100 + 100) + (100 * 50 + 50) + (50 * 10 + 10)
assert sum(p.numel() for p in my_mlp.parameters()) == my_count_by_hand == 84_060
print("Rätt!")

### Övning 2
Ett bredare nät och fler epoker räcker. Fler epoker ger modellen tid att lära sig mer. Vi tittar på val accuracy för att se när det slutar förbättras (mer om det i L06–L07).

In [ ]:
set_seed(42)
train_loader, val_loader, _ = fashion_mnist_loaders()
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 512), nn.ReLU(),
    nn.Linear(512, 256), nn.ReLU(),
    nn.Linear(256, 10),
).to(device)
history = fit(model, train_loader, val_loader, nn.CrossEntropyLoss(),
              torch.optim.Adam(model.parameters(), lr=1e-3), device, epochs=10)
assert max(history["val_acc"]) >= 0.88, max(history["val_acc"])
print("Rätt!", max(history["val_acc"]))

### Övning 3 – förslag på svar
1. Det sista lagret ska ge **logits**, alltså obegränsade tal, till `CrossEntropyLoss`. En ReLU skulle klippa bort negativa logits, och modellen kan då inte uttrycka att en klass är osannolik.
2. **Försvinnande gradienter:** sigmoidens derivata är högst 0,25, och produkten över 10 lager blir pytteliten. De första lagren lär sig nästan ingenting. Byt till ReLU.
3. För stora bilder: en MLP på 1 000 × 1 000 pixlar (× 3 färger) behöver 3 miljoner vikter *per neuron* i första lagret. Den vet dessutom inte att närliggande pixlar hör ihop, och ett mönster som flyttas en pixel ser helt nytt ut. Det löser CNN (L10). En MLP är också ett sämre val för sekvenser av varierande längd (L12–L13).